In [ ]:
#  Misinformation detection ladder  ·  English / isiZulu / Sepedi = Loads ONE labelled JSON file, (optionally) cleans the text, collapses the label to binary (misinformation vs rest), splits 80/10/10 -- separate in original order (stratified on label x language) THEN shuffle each segment by seed -- and runs a ladder of models on the SAME split:

# classical (TF-IDF + LogReg / SVM / NB)
#        -> MzansiLM-125M            (decoder-only, SA)
#        -> AfroXLMR-base-76L         (encoder)
#        -> AfroXLMR-large-76L        (encoder, the model you keep)

#  0.) ENVIRONMENT  (pins Transformers 4.x for MzansiLM; restarts the runtime once)

import os, sys, subprocess

_SENTINEL = "/content/.misinfo_env_ready"

def _needs_env_setup():
    # Set up if the marker is missing OR Transformers 5 is present (MzansiLM's Llama config is rejected by v5's strict validation, so we pin 4.x).
    if not os.path.exists(_SENTINEL):
        return True
    try:
        import transformers
        return int(transformers.__version__.split(".")[0]) >= 5
    except Exception:
        return True

if _needs_env_setup():
    print(">>> Preparing environment: pinning Transformers 4.x (restarts once) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers>=4.52.4,<5", "accelerate", "peft"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--force-reinstall", "--no-cache-dir", "pyarrow"], check=False)
    open(_SENTINEL, "w").close()
    print("\n*** Environment ready. RESTARTING now - after it restarts, RE-RUN "
          "this same cell. ***")
    os.kill(os.getpid(), 9)

import json, random, inspect, re, unicodedata, datetime as _dt
from dataclasses import dataclass, field
from collections import Counter
import numpy as np
from scipy.special import softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             accuracy_score, precision_recall_fscore_support)

#1. CONFIG = everything experimentable lives here

@dataclass
class DataCfg:
    file: str = "translated_masked_samples_combined.json"
    split: tuple = (0.80, 0.10, 0.10)   # train / val / test (must sum to 1.0)
    text_fields:  tuple = ("statement", "text", "Text", "content", "tweet")
    label_fields: tuple = ("classification", "label", "label_str", "class", "category")
    lang_fields:  tuple = ("language", "lang")
    clean_text: bool = True
    breakdown_fields: tuple = ("language",)
    breakdown_top_n: int = 25           # cap console rows for high-cardinality dims
    stratify_fields: tuple = ("language",)
    min_chars: int = 3            # drop near-empty rows
    max_token_repeat: int = 15    # drop degenerate rows (a word repeated >= N times)
    max_chars: int = 5000         # safety cap on pathological rows

@dataclass
class TaskCfg:
    mode: str = "binary"                # "auto" | "binary" | "multiclass"
    pos_class: str = "misinformation"   # positive class; everything else -> negative

@dataclass
class TrainCfg:
    seed: int = 777
    max_length: int = 256              # token cap (texts here run ~p99 121 words)
    weight_decay: float = 0.01
    select_metric: str = "f1"          # "auc" | "f1"
    early_stop_metric: str = "f1"      # informational only; always mirrors select_metric
    early_stop_patience: int = 2
    # classical baselines
    classical: tuple = ("logreg", "svm", "naive_bayes")
    tfidf_ngram: tuple = (1, 2)
    tfidf_min_df: int = 2

@dataclass
class OutputCfg:
    metrics_path: str = "misinfo_metrics.json"
    drive_subdir: str = "afroxlmr76L_misinfo"   # Google Drive folder stem

@dataclass
class TFModel:
    """One transformer rung of the ladder. Add / remove / reorder freely."""
    key: str                    # metrics_report key
    label: str                  # header / prompt name
    short: str                  # compact name for the comparison table
    hf_name: str                # Hugging Face repo id
    batch: int
    lr: float
    epochs: int = 10
    grad_accum: int = 1
    grad_ckpt: bool = False
    warmup: float = 0.1
    lora: bool = False
    save: bool = False          # keep artifacts + offer Drive/download?
    gated: bool = True          # ask y/N before running?
    requires_tf4: bool = False  # skip (don't crash) if Transformers >= 5
    kind: str = "encoder"       # informational
    note: str = ""

# the model ladder
TRANSFORMERS = [
    TFModel("mzansilm", "MzansiLM-125M", "MzansiLM",
            "anrilombard/mzansilm-125m",
            batch=16, lr=2e-5, epochs=10, grad_accum=1, grad_ckpt=False, warmup=0.1,
            lora=True,
            kind="decoder", requires_tf4=True,
            note="UCT decoder-only, 11 SA languages"),
    TFModel("afroxlmr_base", "AfroXLMR-base-76L_script", "AfroXLMR-base",
            "Davlan/afro-xlmr-base-76L_script",
            batch=16, lr=2e-5, epochs=10, grad_accum=1, grad_ckpt=False, warmup=0.1,
            lora=True,
            note="XLM-R base, 76 languages (+N'Ko/Tifinagh)"),
    TFModel("afroxlmr", "AfroXLMR-large-76L", "AfroXLMR-lg",
            "Davlan/afro-xlmr-large-76L",
            batch=8, lr=1e-5, epochs=10, grad_accum=2, grad_ckpt=True, warmup=0.15,
            lora=True, save=True, note="the model you keep"),
]

DATA, TASK, TRAIN, OUT = DataCfg(), TaskCfg(), TrainCfg(), OutputCfg()

SEED = TRAIN.seed
random.seed(SEED); np.random.seed(SEED)
# best-epoch / early-stopping metric: "auc" or "f1" (see TRAIN.select_metric).
_SELECT       = TRAIN.select_metric.lower().replace("eval_", "")
assert _SELECT in ("auc", "f1"), \
    f"TRAIN.select_metric must be 'auc' or 'f1' (got {TRAIN.select_metric!r})"
_BEST_KEY     = f"eval_{_SELECT}"           # 'eval_auc' or 'eval_f1'
_BEST_GREATER = True                        # both AUC and F1: higher is better
_SELECT_LABEL = _SELECT.upper()             # 'AUC' / 'F1' for prints + report

# these are populated once the data is loaded (the pool defines the label space)
CLASS_NAMES = None; NUM_LABELS = None; POS_INDEX = None; AVG = None; _CLASS_IDX = None



# 2.) data loading, label encoding

def _ask_yes_no(question):
    try:
        return input(f"\n{question} [y/N]: ").strip().lower() in ("y", "yes")
    except EOFError:
        return False

def _first_field(row, candidates):
    for k in candidates:
        v = row.get(k)
        if isinstance(v, str) and v.strip():
            return v
    return ""

# Text cleaning ported from ExtractDataSet.py (emoji/punct strip, greeting chop, lowercase) so raw GeneratedSamples text matches the cleaned dataset
_EMOJI = re.compile(
    "[\U0001F1E6-\U0001F1FF\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF\U0001F700-\U0001FAFF\U00002600-\U000026FF"
    "\U00002700-\U000027BF\U0001F000-\U0001F0FF\U0000FE0F\U0000200D]+")
_EXCL = re.compile(r"!+")
_MIN_OPENING_WORDS = 10

_PUNCT = set(r"""!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~""")

def _clean_text(text, *,
                strip_emojis=True,
                exclamations_to_periods=True,
                strip_punctuation=True,
                keep_periods=True,
                chop_greeting=True,
                chop_short_openings=True,
                lowercase=True):
    # Single configurable cleaner
    keep_dot = keep_periods or chop_greeting or chop_short_openings
    if strip_emojis:
        text = _EMOJI.sub("", text)
    if exclamations_to_periods:
        text = _EXCL.sub(".", text)
    if strip_punctuation:
        text = "".join(c for c in text
                       if (keep_dot and c == ".")
                       or (c not in _PUNCT and not unicodedata.category(c).startswith("P")))
    text = re.sub(r"\s+", " ", text).strip()
    if chop_greeting and text.lower().startswith("dumelang") and "." in text:
        text = text.split(".", 1)[1].strip()                     # leftover greeting
    if chop_short_openings:
        while "." in text:                                       # short opening lines
            first, rest = text.split(".", 1)
            if len(first.split()) >= _MIN_OPENING_WORDS:
                break
            text = rest.strip()
    text = text.strip()
    return text.lower() if lowercase else text

def _get_ci(row, field):
    # Case-insensitive field lookup ('Stage'/'Domain' -> stage/domain)
    if field in row:
        v = row[field]
    else:
        lk = {k.lower(): k for k in row}
        v = row.get(lk.get(field.lower(), field))
    return str(v).strip() if v is not None and str(v).strip() else "Unknown"

def _is_degenerate(text):
    toks = text.split()
    return (not toks) or Counter(toks).most_common(1)[0][1] >= DATA.max_token_repeat

# fields to carry per row = everything we slice or stratify by
_META_FIELDS = tuple(dict.fromkeys(DATA.breakdown_fields + DATA.stratify_fields))

def _load_records(raw_records, *, tag):
    # Raw JSON rows -> list of {text, raw_label, meta:{field:value}} with cleaning + hygiene. `meta` carries every breakdown/stratify field.
    rows, empty, degen = [], 0, 0
    for row in raw_records:
        stmt = _first_field(row, DATA.text_fields).strip()
        cls  = _first_field(row, DATA.label_fields).strip()
        if DATA.clean_text and stmt:
            stmt = _clean_text(stmt)
        if len(stmt) < DATA.min_chars or not cls:
            empty += 1; continue
        if _is_degenerate(stmt):
            degen += 1; continue
        if len(stmt) > DATA.max_chars:
            stmt = stmt[:DATA.max_chars]
        meta = {f: _get_ci(row, f) for f in _META_FIELDS}
        rows.append({"text": stmt, "raw_label": cls.lower(), "meta": meta})
    print(f"[{tag}] kept {len(rows)}  (dropped {empty} empty, {degen} degenerate)")
    return rows

def _encode_label(c):
    if AVG == "binary":
        return 1 if c == TASK.pos_class else 0
    return _CLASS_IDX.get(c, None)

def _lang_group(lg):
    return "English" if str(lg).strip().lower() == "english" else "African"


#  3.) metrics + breakdowns
def _auc(y_true, prob_matrix):
    y_true = np.asarray(y_true)
    if len(set(y_true.tolist())) < 2:
        return None
    try:
        if NUM_LABELS == 2:
            return float(roc_auc_score(y_true, prob_matrix[:, 1]))
        if len(set(y_true.tolist())) < NUM_LABELS:
            return None
        return float(roc_auc_score(y_true, prob_matrix, multi_class="ovr", average="macro"))
    except ValueError:
        return None

def _split_metrics(y_true, y_pred, prob_matrix):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=AVG, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_LABELS)))
    rep = classification_report(y_true, y_pred, labels=list(range(NUM_LABELS)),
                                target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    wrong = int((y_pred != y_true).sum()); total = int(len(y_true))
    out = {"averaging": AVG, "accuracy": float(accuracy_score(y_true, y_pred)),
           "precision": float(p), "recall": float(r), "f1": float(f1),
           "roc_auc": _auc(y_true, prob_matrix),
           "confusion_matrix": cm.tolist(),
           "confusion_matrix_labels": {"rows": "true", "cols": "pred", "order": CLASS_NAMES},
           "n_samples": total, "n_wrong": wrong, "n_correct": total - wrong,
           "error_rate": wrong / total if total else None,
           "classification_report": rep}
    key = CLASS_NAMES[POS_INDEX]
    if key in rep:
        out["misinformation_class"] = {"precision": rep[key]["precision"],
                                       "recall": rep[key]["recall"],
                                       "f1": rep[key]["f1-score"],
                                       "support": rep[key]["support"]}
    return out

def _subset_metrics(y_true, y_pred, prob_matrix):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    total = int(len(y_true))
    if total == 0:
        return {"n_samples": 0, "note": "no samples"}
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=AVG, zero_division=0)
    single_class = len(set(y_true.tolist())) < 2      # f1/precision/recall undefined
    out = {"accuracy": float(accuracy_score(y_true, y_pred)),
           "precision": None if single_class else float(p),
           "recall": None if single_class else float(r),
           "f1": None if single_class else float(f1), "n_samples": total,
           "n_wrong": int((y_pred != y_true).sum()),
           "error_rate": float((y_pred != y_true).mean()),
           "pred_pos_rate": float((y_pred == POS_INDEX).mean()),   # % predicted misinfo
           "roc_auc": _auc(y_true, prob_matrix)}
    mask = (y_true == POS_INDEX)
    if mask.sum() > 0:
        out["misinfo_support"] = int(mask.sum())
        out["misinfo_recall"] = float((y_pred[mask] == POS_INDEX).mean())
    return out

def _breakdown(y_true, y_pred, prob_matrix, by_arr, groups=None):
    # Per-value metrics for one metadata field, sorted by support (desc).
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    prob_matrix = np.asarray(prob_matrix); by_arr = np.asarray(by_arr)
    keyed = [(k, _subset_metrics(y_true[by_arr == k], y_pred[by_arr == k], prob_matrix[by_arr == k]))
             for k in set(by_arr.tolist())]
    keyed.sort(key=lambda kv: -kv[1].get("n_samples", 0))
    out = {k: m for k, m in keyed}
    if groups:
        grp = np.array([groups(v) for v in by_arr], dtype=object)
        out["_grouped"] = {g: _subset_metrics(y_true[grp == g], y_pred[grp == g], prob_matrix[grp == g])
                           for g in sorted(set(grp.tolist()))}
    return out

def _attach_breakdowns(metrics, y_true, y_pred, probm, meta_slice):
    # Attach a per-value breakdown for every configured field.
    bd = {}
    for field in DATA.breakdown_fields:
        arr = meta_slice.get(field)
        if arr is None or len(set(np.asarray(arr).tolist())) < 2:
            continue
        grp = _lang_group if field.lower() in ("language", "lang") else None
        bd[field] = _breakdown(y_true, y_pred, probm, arr, groups=grp)
    if bd:
        metrics["breakdowns"] = bd
    return metrics

def _print_breakdowns(title_prefix, metrics):
    for field, bd in metrics.get("breakdowns", {}).items():
        print("\n" + "-" * 72)
        print(f"{title_prefix} · by {field}")
        print("-" * 72)
        print(f"{field[:20]:<22}{'n':>6}{'acc':>8}{'f1':>8}{'%mis':>8}{'auc':>8}")
        rows = [(k, m) for k, m in bd.items() if k != "_grouped"]
        shown = rows[:DATA.breakdown_top_n]
        for k, m in shown:
            if not m or m.get("n_samples", 0) == 0:
                continue
            auc = m.get("roc_auc"); auc_s = f"{auc:.3f}" if auc is not None else "  n/a"
            f1 = m.get("f1"); f1_s = f"{f1:.3f}" if f1 is not None else "  n/a"
            pm = m.get("pred_pos_rate"); pm_s = f"{pm*100:.0f}%" if pm is not None else " n/a"
            print(f"{str(k)[:20]:<22}{m['n_samples']:>6}{m['accuracy']:>8.3f}{f1_s:>8}{pm_s:>8}{auc_s:>8}")
        if len(rows) > len(shown):
            print(f"  ... (+{len(rows)-len(shown)} more values in the metrics JSON)")
        if "_grouped" in bd:
            print("  -- grouped --")
            for g, m in bd["_grouped"].items():
                if m.get("n_samples", 0) == 0:
                    continue
                auc = m.get("roc_auc"); auc_s = f"{auc:.3f}" if auc is not None else "  n/a"
                f1 = m.get("f1"); f1_s = f"{f1:.3f}" if f1 is not None else "  n/a"
                pm = m.get("pred_pos_rate", 0.0)
                print(f"{g:<22}{m['n_samples']:>6}{m['accuracy']:>8.3f}{f1_s:>8}"
                      f"{pm*100:>7.0f}%{auc_s:>8}")

def _print_split(title, m):
    print("\n" + "=" * 66); print(title); print("=" * 66)
    auc = m["roc_auc"]; auc_s = f"{auc:.3f}" if auc is not None else "n/a"
    print(f"accuracy={m['accuracy']:.3f}  P({m['averaging']})={m['precision']:.3f}  "
          f"R={m['recall']:.3f}  F1={m['f1']:.3f}  auc={auc_s}")
    mc = m.get("misinformation_class")
    if mc:
        print(f"MISINFO class -> precision={mc['precision']:.3f} recall={mc['recall']:.3f} "
              f"f1={mc['f1']:.3f} (support={int(mc['support'])})")
    print(f"confusion [rows=true, cols=pred] order={CLASS_NAMES}:")
    for row in m["confusion_matrix"]:
        print("   ", row)
    print(f"wrong: {m['n_wrong']}/{m['n_samples']} ({m['error_rate']*100:.1f}%)")

def _write_metrics(report):
    with open(OUT.metrics_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f"\nMetrics written -> {OUT.metrics_path}")

def _offer_download(path, label):
    if _ask_yes_no(f"Download the {label}?"):
        try:
            from google.colab import files as _dl
            _dl.download(path)
        except ImportError:
            print(f"(Not on Colab) File is at: {path}")
        except Exception as e:
            print(f"[!] Download failed ({e}). File is at {path}.")



#  4.) load -> discover classes -> split
def load_and_split():
    # Load ONE file, clean, discover classes, split 80/10/10 stratified on label (and language when present). Returns (D, meta) and sets CLASS_* globals.
    global CLASS_NAMES, NUM_LABELS, POS_INDEX, AVG, _CLASS_IDX

    # upload / read the file
    try:
        from google.colab import files
        print(f">>> Upload the dataset JSON (e.g. {DATA.file}):")
        up = files.upload()
        jsons = [n for n in up if n.lower().endswith(".json")]   # never grab a .py etc.
        stem = DATA.file.split(".")[0].lower()
        name = (DATA.file if DATA.file in up
                else next((n for n in jsons if stem in n.lower()), None)
                or (jsons[0] if jsons else None))
        if name is None:
            raise SystemExit(f"[FATAL] No .json file uploaded (got: {list(up)}). "
                             f"Upload the dataset, not a script or other file.")
        blob = up[name].decode("utf-8")
    except ImportError:
        name = DATA.file
        blob = open(DATA.file, encoding="utf-8").read()
    try:
        raw = json.loads(blob)
    except json.JSONDecodeError as e:
        raise SystemExit(f"[FATAL] '{name}' is not valid JSON ({e}). Make sure you "
                         f"uploaded the dataset file, not a .py script or a partial file.")
    print(f"\n'{name}': {len(raw)} rows")

    # clean 
    rows = _load_records(raw, tag=name)
    if not rows:
        first = raw[0] if raw and isinstance(raw[0], dict) else {}
        gt, gl = _first_field(first, DATA.text_fields), _first_field(first, DATA.label_fields)
        hint = ("\n    -> text field exists but NO label field: looks UNLABELED "
                "(e.g. a raw scrape); can't be used for supervised training."
                if gt and not gl else "")
        raise SystemExit(
            f"[FATAL] '{name}' produced 0 usable rows "
            f"({'text FOUND' if gt else 'text MISSING'}, "
            f"{'label FOUND' if gl else 'label MISSING'}).\n"
            f"    First record's keys: {sorted(first.keys()) if first else 'n/a'}.{hint}\n"
            f"    Fix: check DATA.text_fields / DATA.label_fields.")

    #  discover the class set 
    present = sorted(set(r["raw_label"] for r in rows))
    mode = ("binary" if len(present) == 2 else "multiclass") if TASK.mode == "auto" else TASK.mode
    if mode == "binary":
        if TASK.pos_class not in present:
            raise SystemExit(f"[FATAL] pos_class '{TASK.pos_class}' not in {present}.")
        neg = [c for c in present if c != TASK.pos_class]
        CLASS_NAMES = [neg[0] if len(neg) == 1 else f"non-{TASK.pos_class}", TASK.pos_class]
        NUM_LABELS, POS_INDEX, AVG = 2, 1, "binary"
    else:
        CLASS_NAMES = present; NUM_LABELS = len(present); AVG = "macro"
        if TASK.pos_class not in CLASS_NAMES:
            raise SystemExit(f"[FATAL] pos_class '{TASK.pos_class}' not found in {CLASS_NAMES}.")
        POS_INDEX = CLASS_NAMES.index(TASK.pos_class)
    _CLASS_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

    texts, ys, metas, unknown = [], [], [], Counter()
    for r in rows:
        yi = _encode_label(r["raw_label"])
        if yi is None:
            unknown[r["raw_label"]] += 1; continue
        texts.append(r["text"]); ys.append(yi); metas.append(r["meta"])
    if unknown:
        print("[!] dropped rows with unseen classes: " +
              ", ".join(f"{k}={v}" for k, v in unknown.most_common()))
    X = np.array(texts, dtype=object)
    Y = np.array(ys, dtype="int64")
    META = {f: np.array([m[f] for m in metas], dtype=object) for f in _META_FIELDS}
    print(f"\nTask: {mode}  |  classes ({NUM_LABELS}): {CLASS_NAMES}  |  POS='{TASK.pos_class}'")
    u, c = np.unique(Y, return_counts=True)
    print("Class dist (binary): " + ", ".join(f"{CLASS_NAMES[i]}={n}" for i, n in zip(u, c)))
    for f in DATA.breakdown_fields:
        if f in META:
            vc = Counter(META[f].tolist())
            head = ", ".join(f"{k}={v}" for k, v in vc.most_common(6))
            more = f" (+{len(vc)-6} more)" if len(vc) > 6 else ""
            print(f"  {f}: {len(vc)} distinct -> {head}{more}")

    # ---- SEPARATE first (in original order, stratified on stratify_fields), then shuffle each segment by seed. No global pre-shuffle decides which rows land in which split.
    tr_f, va_f, te_f = DATA.split
    assert abs(tr_f + va_f + te_f - 1.0) < 1e-9, "DATA.split must sum to 1.0"

    strat_fields = [f for f in DATA.stratify_fields if f in META]
    if strat_fields:
        keys = list(zip(*([Y.tolist()] + [META[f].tolist() for f in strat_fields])))
    else:
        keys = list(map(int, Y.tolist()))
    strata = {}
    for i, k in enumerate(keys):
        strata.setdefault(k, []).append(i)          # original order preserved

    tr_idx, va_idx, te_idx = [], [], []
    for members in strata.values():
        n = len(members)
        n_tr = int(n * tr_f); n_va = int(n * va_f)   # floor; remainder -> test
        if n >= 3:                                   # keep val/test non-empty
            n_va = max(1, n_va)
            if n - n_tr - n_va < 1:
                n_tr = n - n_va - 1
        tr_idx += members[:n_tr]
        va_idx += members[n_tr:n_tr + n_va]
        te_idx += members[n_tr + n_va:]

    tr_idx = np.array(tr_idx, dtype=int); va_idx = np.array(va_idx, dtype=int)
    te_idx = np.array(te_idx, dtype=int)
    _rng = np.random.default_rng(SEED)               # shuffle EACH segment by seed
    _rng.shuffle(tr_idx); _rng.shuffle(va_idx); _rng.shuffle(te_idx)

    def meta_slice(ix):
        return {f: META[f][ix] for f in _META_FIELDS}
    D = {"x_train": X[tr_idx], "y_train": Y[tr_idx], "meta_train": meta_slice(tr_idx),
         "x_eval":  X[va_idx], "y_eval":  Y[va_idx], "meta_eval":  meta_slice(va_idx),
         "x_infer": X[te_idx], "y_infer": Y[te_idx], "meta_infer": meta_slice(te_idx)}

    print(f"\nSplit (80/10/10) -> TRAIN:{len(tr_idx)}  VAL:{len(va_idx)}  TEST:{len(te_idx)}")
    for nm, k in [("TRAIN", "train"), ("VAL", "eval"), ("TEST", "infer")]:
        y = D[f"y_{k}"]
        cl = {CLASS_NAMES[i]: int((y == i).sum()) for i in range(NUM_LABELS)}
        extra = ""
        if "language" in D[f"meta_{k}"]:
            lg = D[f"meta_{k}"]["language"]
            extra = "  lang=" + str({str(l): int((lg == l).sum()) for l in sorted(set(lg.tolist()))})
        print(f"  {nm:<6} class={cl}{extra}")

    meta = {"timestamp_utc": _dt.datetime.now(_dt.timezone.utc).isoformat(),
            "seed": SEED, "task_mode": mode, "classes": CLASS_NAMES, "pos_class": TASK.pos_class,
            "select_metric": _SELECT,
            "data_layout": {"file": name, "clean_text": DATA.clean_text,
                            "split": {"train": tr_f, "val": va_f, "test": te_f},
                            "split_method": "separate in original order (stratified on "
                                            + " x ".join(strat_fields or ["label"])
                                            + "), then shuffle each segment by seed",
                            "breakdown_fields": list(DATA.breakdown_fields)},
            "dataset": {"total_usable": int(len(X)), "train_rows": int(len(tr_idx)),
                        "val_rows": int(len(va_idx)), "test_rows": int(len(te_idx)),
                        "class_distribution": {CLASS_NAMES[i]: int(n) for i, n in zip(u, c)}},
            "max_length": TRAIN.max_length}
    return D, meta


#  5.) CLASSICAL BASELINES: TF-IDF + LogReg / SVM / NB
def _clf_prob_matrix(model, Xm):
    #Class-probability matrix aligned to CLASS_NAMES (predict_proba, else softmax over decision_function for LinearSVC).
    full = np.zeros((Xm.shape[0], NUM_LABELS))
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(Xm)
    else:
        d = model.decision_function(Xm)
        prob = softmax(np.column_stack([-d, d]) if d.ndim == 1 else d, axis=1)
    for j, ci in enumerate(model.classes_):
        full[:, int(ci)] = prob[:, j]
    return full

def _clf_metrics(model, Xm, y_true, meta_slice=None):
    probm = _clf_prob_matrix(model, Xm)
    pred = probm.argmax(axis=1)
    m = _split_metrics(y_true, pred, probm)
    if meta_slice is not None:
        _attach_breakdowns(m, y_true, pred, probm, meta_slice)
    return m

def run_classical(D, report):
    print("\n" + "#" * 66)
    print("# BENCHMARK - TF-IDF classical baselines")
    print("#" * 66)
    tfidf = TfidfVectorizer(ngram_range=TRAIN.tfidf_ngram, min_df=TRAIN.tfidf_min_df, sublinear_tf=True)
    Xtr = tfidf.fit_transform(D["x_train"])
    Xev = tfidf.transform(D["x_eval"]); Xin = tfidf.transform(D["x_infer"])
    builders = {
        "logreg": ("TF-IDF + LogisticRegression(balanced)",
                   lambda: LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)),
        "svm": ("TF-IDF + LinearSVC(balanced)",
                lambda: LinearSVC(class_weight="balanced", random_state=SEED)),
        "naive_bayes": ("TF-IDF + MultinomialNB", lambda: MultinomialNB()),
    }
    report["classical_baselines"] = {}
    test_metrics = {}
    for key in TRAIN.classical:
        desc, make = builders[key]
        model = make(); model.fit(Xtr, D["y_train"])
        ev = _clf_metrics(model, Xev, D["y_eval"])
        te = _clf_metrics(model, Xin, D["y_infer"], D["meta_infer"])
        _print_split(f"{desc} - VAL", ev)
        _print_split(f"{desc} - TEST (untouched)", te)
        _print_breakdowns(f"{key} TEST", te)
        report["classical_baselines"][key] = {"model": desc, "val_split": ev, "test_split": te}
        test_metrics[key] = te
    return test_metrics



#  6.) TRANSFORMERS: shared trainer + one run() per registry entry
def _make_callbacks():
    # Early stopping + best-checkpoint selection on the SAME metric (metric_for_best_model = _BEST_KEY, higher is better). 
    from transformers import EarlyStoppingCallback
    return [EarlyStoppingCallback(early_stopping_patience=TRAIN.early_stop_patience)]

def train_transformer(spec, D, save_dir=None):
    # Fine-tune one TFModel on the shared split. Returns (eval_metrics, infer_metrics, epoch_history, class_weights, saved_dir).
    import torch, torch.nn as nn
    from torch.utils.data import Dataset as TorchDataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                              TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
    torch.manual_seed(SEED); set_seed(SEED)

    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f"\n>>> [{spec.hf_name}] GPU: {torch.cuda.get_device_name(0)} | "
              f"VRAM: {p.total_memory/1e9:.1f} GB")
    else:
        print(f"\n[!] [{spec.hf_name}] No GPU. Runtime -> Change runtime type -> T4 GPU.")

    tok = AutoTokenizer.from_pretrained(spec.hf_name)
    if tok.pad_token_id is None:                      # decoder-only models
        tok.pad_token = tok.eos_token or tok.unk_token
    tok.padding_side = "right"

    class DS(TorchDataset):
        def __init__(self, x, y):
            self.enc = tok(list(x), truncation=True, max_length=TRAIN.max_length)
            self.labels = [int(v) for v in y]
        def __len__(self): return len(self.labels)
        def __getitem__(self, i):
            item = {k: self.enc[k][i] for k in self.enc}; item["labels"] = self.labels[i]; return item

    tr = DS(D["x_train"], D["y_train"]); ev = DS(D["x_eval"], D["y_eval"]); inf = DS(D["x_infer"], D["y_infer"])
    id2label = {i: c for i, c in enumerate(CLASS_NAMES)}
    label2id = {c: i for i, c in enumerate(CLASS_NAMES)}
    model = AutoModelForSequenceClassification.from_pretrained(
        spec.hf_name, num_labels=NUM_LABELS, id2label=id2label, label2id=label2id)
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tok.pad_token_id

    if spec.grad_ckpt:
        model.config.use_cache = False
        model.gradient_checkpointing_enable()
        if hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    if spec.lora:
        from peft import LoraConfig, get_peft_model, TaskType
        model = get_peft_model(model, LoraConfig(
            task_type=TaskType.SEQ_CLS, r=64, lora_alpha=128,
            lora_dropout=0.1, bias="none"))
        model.print_trainable_parameters()

    counts = np.bincount(D["y_train"], minlength=NUM_LABELS).astype(float); counts[counts == 0] = 1.0
    cw = torch.tensor(len(D["y_train"]) / (NUM_LABELS * counts), dtype=torch.float)

    class WTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kw):
            lb = inputs.pop("labels"); out = model(**inputs)
            loss = nn.CrossEntropyLoss(weight=cw.to(out.logits.device))(
                out.logits.view(-1, NUM_LABELS), lb.view(-1))
            return (loss, out) if return_outputs else loss

    def cmetrics(ep):
        logits, y_ = ep; preds = logits.argmax(axis=1)
        p, r, f1, _ = precision_recall_fscore_support(y_, preds, average=AVG, zero_division=0)
        _, _, per_f1, _ = precision_recall_fscore_support(
            y_, preds, labels=list(range(NUM_LABELS)), average=None, zero_division=0)
        out = {"accuracy": accuracy_score(y_, preds), "precision": p, "recall": r,
               "f1": f1, "misinfo_f1": float(per_f1[POS_INDEX])}
        auc = _auc(y_, softmax(logits, axis=1))
        # always emit 'auc' AND 'f1' -> 'eval_auc'/'eval_f1' exist every epoch, so either can drive selection/early stopping; AUC falls back to chance if a degenerate eval batch made it undefined
        out["auc"] = float(auc) if auc is not None else 0.5
        return out

    supported = set(inspect.signature(TrainingArguments.__init__).parameters)
    common = dict(output_dir="hf_out", num_train_epochs=spec.epochs,
                  per_device_train_batch_size=spec.batch,
                  per_device_eval_batch_size=max(16, spec.batch),
                  gradient_accumulation_steps=spec.grad_accum, gradient_checkpointing=spec.grad_ckpt,
                  learning_rate=spec.lr, weight_decay=TRAIN.weight_decay, logging_steps=20,
                  save_total_limit=1, load_best_model_at_end=True, metric_for_best_model=_BEST_KEY,
                  greater_is_better=_BEST_GREATER, seed=SEED,
                  fp16=torch.cuda.is_available(), report_to="none")
    if "eval_strategy" in supported: common["eval_strategy"] = "epoch"
    elif "evaluation_strategy" in supported: common["evaluation_strategy"] = "epoch"
    if "save_strategy" in supported: common["save_strategy"] = "epoch"
    if "warmup_ratio" in supported: common["warmup_ratio"] = spec.warmup
    elif "warmup_steps" in supported: common["warmup_steps"] = 100
    args = TrainingArguments(**{k: v for k, v in common.items() if k in supported})

    tok_kw = ("processing_class"
              if "processing_class" in inspect.signature(Trainer.__init__).parameters else "tokenizer")
    trainer = WTrainer(model=model, args=args, train_dataset=tr, eval_dataset=ev,
                       data_collator=DataCollatorWithPadding(tok), compute_metrics=cmetrics,
                       callbacks=_make_callbacks(), **{tok_kw: tok})
    trainer.train()

    def history(log):
        pe = {}
        for e in log:
            ep = e.get("epoch")
            if ep is None: continue
            row = pe.setdefault(round(float(ep), 4), {"epoch": round(float(ep), 4)})
            for k, v in e.items():
                if k == "loss": row["training_loss"] = v
                elif k.startswith("eval_"): row[k.replace("eval_", "val_")] = v
        return [pe[k] for k in sorted(pe)]

    def predict(ds):
        out = trainer.predict(ds); logits = out.predictions
        return logits.argmax(axis=1), softmax(logits, axis=1), np.array(out.label_ids)

    ep_p, ep_pb, ep_y = predict(ev); in_p, in_pb, in_y = predict(inf)
    em = _split_metrics(ep_y, ep_p, ep_pb)
    im = _split_metrics(in_y, in_p, in_pb)
    _attach_breakdowns(im, in_y, in_p, in_pb, D["meta_infer"])

    saved = None
    if save_dir is not None:
        (model.save_pretrained if spec.lora else trainer.save_model)(save_dir)
        tok.save_pretrained(save_dir); saved = save_dir
        print(f"Saved model -> {save_dir}/")
    return em, im, history(trainer.state.log_history), cw.tolist(), saved

def run_transformer(spec, D, report):
    """Gate -> (version guard) -> train -> record. Returns (test_metrics, saved_dir)."""
    if spec.gated and not _ask_yes_no(f"Fine-tune {spec.label}? (No = skip)"):
        print(f"\nSkipping {spec.label}.")
        return None, None
    print("\n" + "#" * 66)
    print(f"# {spec.label}  ({spec.kind}; {spec.note})")
    print("#" * 66)

    if spec.requires_tf4:
        import transformers as _tf
        if int(_tf.__version__.split(".")[0]) >= 5:
            print(f"[!] Transformers {_tf.__version__} installed but {spec.label} needs 4.x. "
                  "Restart runtime and re-run this cell. Skipping.")
            report[spec.key] = {"model": spec.hf_name, "error": f"transformers {_tf.__version__} >= 5"}
            return None, None

    save_dir = f"{spec.key}_misinfo" if spec.save else None
    try:
        em, im, hist, cw, saved = train_transformer(spec, D, save_dir)
    except Exception as e:
        print(f"\n[!] {spec.label} training failed ({type(e).__name__}: {e}). Continuing.")
        report[spec.key] = {"model": spec.hf_name, "error": f"{type(e).__name__}: {e}"}
        return None, None

    _print_split(f"{spec.label} - VAL (early stopping; not clean hold-out)", em)
    _print_split(f"{spec.label} - TEST (untouched)", im)
    _print_breakdowns(f"{spec.short} TEST", im)
    report[spec.key] = {
        "model": spec.hf_name, "kind": spec.kind, "use_lora": spec.lora,
        "config": {"epochs_max": spec.epochs, "learning_rate": spec.lr,
                   "per_device_train_batch_size": spec.batch,
                   "gradient_accumulation_steps": spec.grad_accum,
                   "effective_batch_size": spec.batch * spec.grad_accum,
                   "warmup_ratio": spec.warmup, "weight_decay": TRAIN.weight_decay,
                   "max_length": TRAIN.max_length, "gradient_checkpointing": spec.grad_ckpt,
                   "fp16": True, "early_stopping_patience": TRAIN.early_stop_patience,
                   "termination": f"stop when '{_BEST_KEY}' does not improve for "
                                  f"{TRAIN.early_stop_patience} epochs",
                   "winning_epoch_metric": f"highest {_BEST_KEY}",
                   "metric_for_best_model": _BEST_KEY},
        "class_weights": cw, "per_epoch_history": hist,
        "val_split_note": "used for early stopping; NOT a clean hold-out", "val_split": em,
        "test_split_note": "untouched hold-out", "test_split": im}
    _write_metrics(report)
    return im, saved



#  7.) SAVE
def save_artifacts(spec, saved_dir):
    if not _ask_yes_no("Upload this run (model + metrics) to Google Drive?"):
        print("Skipped Drive upload.")
    else:
        try:
            from google.colab import drive
            import shutil
            drive.mount("/content/drive")
            stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
            tag = "lora" if spec.lora else "full"
            dest = f"/content/drive/MyDrive/{OUT.drive_subdir}_{tag}_seed{SEED}_{stamp}"
            n = 1
            while os.path.exists(dest):
                dest = f"{dest}_v{n}"; n += 1
            shutil.copytree(saved_dir, dest); shutil.copy(OUT.metrics_path, dest)
            print(f"Uploaded to Drive -> {os.path.basename(dest)}/ (model + metrics.json)")
        except Exception as e:
            print(f"[!] Drive upload skipped ({e}). Files are saved locally.")

    _offer_download(OUT.metrics_path, "combined metrics JSON")

    if _ask_yes_no(f"Download the {spec.short} model artifacts?"):
        try:
            import shutil
            zip_path = shutil.make_archive(saved_dir + "_bundle", "zip", saved_dir)
            print(f"Zipped -> {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
            try:
                from google.colab import files as _dl
                _dl.download(zip_path)
            except ImportError:
                print(f"(Not on Colab) Zip is at: {zip_path}")
        except Exception as e:
            print(f"[!] Zip/download failed ({e}). Artifacts are in {saved_dir}/.")
    else:
        print(f"Skipped download. Artifacts remain in {saved_dir}/ until the runtime stops.")


#  8. COMPARISON 
def compare(report, classical_test):
    print("\n" + "=" * 66); print("All models  (TEST split)"); print("=" * 66)
    rows = []
    for key in TRAIN.classical:
        if key in classical_test:
            rows.append((f"TF-IDF+{ {'logreg':'LR','svm':'SVM','naive_bayes':'NB'}[key] }", classical_test[key]))
    for spec in TRANSFORMERS:
        m = report.get(spec.key, {}).get("test_split")
        if m:
            rows.append((spec.short, m))
    print(f"{'model':<14}{'acc':>8}{'f1':>9}{'misfF1':>8}{'auc':>8}")
    for nm, m in rows:
        auc = m["roc_auc"]; auc_s = f"{auc:.3f}" if auc is not None else "  n/a"
        mf = m.get("misinformation_class", {}).get("f1"); mf_s = f"{mf:.3f}" if mf is not None else "  n/a"
        print(f"{nm:<14}{m['accuracy']:>8.3f}{m['f1']:>9.3f}{mf_s:>8}{auc_s:>8}")

    if not classical_test:
        return
    best_c = max(classical_test, key=lambda k: classical_test[k]["f1"])
    ct = classical_test[best_c]["f1"]
    # primary transformer = the save=True spec if it ran, else the best transformer
    tf_ran = {s.key: report[s.key]["test_split"] for s in TRANSFORMERS
              if report.get(s.key, {}).get("test_split")}
    if not tf_ran:
        print(f"\nNo transformer ran; best classical is '{best_c}' (F1={ct:.3f}).")
        return
    primary = next((s for s in TRANSFORMERS if s.save and s.key in tf_ran), None)
    prim_key = primary.key if primary else max(tf_ran, key=lambda k: tf_ran[k]["f1"])
    at = tf_ran[prim_key]["f1"]; gain = at - ct
    prim_label = next(s.short for s in TRANSFORMERS if s.key == prim_key)
    print(f"\nReading (F1): {prim_label} - best classical ({best_c}) = {gain:+.3f}")
    # pairwise deltas among transformers that ran
    for s in TRANSFORMERS:
        if s.key in tf_ran and s.key != prim_key:
            print(f"              {prim_label} - {s.short} = {at - tf_ran[s.key]['f1']:+.3f}")
    if gain >= 0.10:
        print("[OK] Clears the best lexical baseline by a wide margin -> real language "
              "understanding beyond surface cues.")
    elif gain > 0.02:
        print("Beats the best baseline modestly. Consider the small test-set size.")
    else:
        print("[!] ~= best classical: barely beats a lexical baseline. Task may be "
              "surface/shortcut-solvable, or the dataset is too small to separate them.")


def main():
    print(f">>> Best-epoch / early-stopping metric: {_SELECT_LABEL}  (TRAIN.select_metric)")
    report = {}
    D, report["run"] = load_and_split()

    classical_test = run_classical(D, report)
    # keep a top-level alias for the primary classical baseline (backward compat)
    if "logreg" in report["classical_baselines"]:
        report["tfidf_baseline"] = report["classical_baselines"]["logreg"]
    _write_metrics(report)

    saved = {}
    for spec in TRANSFORMERS:
        _, saved_dir = run_transformer(spec, D, report)
        if saved_dir:
            saved[spec.key] = saved_dir

    for spec in TRANSFORMERS:
        if spec.save and spec.key in saved:
            save_artifacts(spec, saved[spec.key])

    compare(report, classical_test)

main()

>>> Best-epoch / early-stopping metric: F1  (TRAIN.select_metric)
>>> Upload the dataset JSON (e.g. translated_masked_samples_combined.json):


Saving translated_masked_samples_combined.json to translated_masked_samples_combined (2).json

'translated_masked_samples_combined (2).json': 13997 rows
[translated_masked_samples_combined (2).json] kept 13877  (dropped 100 empty, 20 degenerate)

Task: binary  |  classes (2): ['non-misinformation', 'misinformation']  |  POS='misinformation'
Class dist (binary): non-misinformation=6869, misinformation=7008
  language: 3 distinct -> English=4661, Sepedi=4642, isiZulu=4574

Split (80/10/10) -> TRAIN:11099  VAL:1385  TEST:1393
  TRAIN  class={'non-misinformation': 5494, 'misinformation': 5605}  lang={'English': 3728, 'Sepedi': 3713, 'isiZulu': 3658}
  VAL    class={'non-misinformation': 686, 'misinformation': 699}  lang={'English': 465, 'Sepedi': 463, 'isiZulu': 457}
  TEST   class={'non-misinformation': 689, 'misinformation': 704}  lang={'English': 468, 'Sepedi': 466, 'isiZulu': 459}

##################################################################
# BENCHMARK - TF-IDF classical baselin

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at Davlan/afro-xlmr-large-76L and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 7,343,106 || all params: 567,235,588 || trainable%: 1.2945


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Misinfo F1,Auc
1,0.689500,0.642841,0.647653,0.669887,0.595136,0.630303,0.630303,0.713619
2,0.435500,0.332015,0.859928,0.879699,0.836910,0.857771,0.857771,0.933392
3,0.422600,0.306255,0.881588,0.858099,0.917024,0.886584,0.886584,0.947482
4,0.289300,0.295830,0.888087,0.875691,0.907010,0.891075,0.891075,0.950176
5,0.257400,0.286796,0.886643,0.888252,0.886981,0.887616,0.887616,0.952859
6,0.235200,0.307043,0.888809,0.869742,0.917024,0.892758,0.892758,0.951050
7,0.292600,0.293700,0.882310,0.900000,0.862661,0.880935,0.880935,0.954232
8,0.273000,0.291896,0.890253,0.881450,0.904149,0.892655,0.892655,0.954284


Saved model -> afroxlmr_misinfo/

AfroXLMR-large-76L - VAL (early stopping; not clean hold-out)
accuracy=0.889  P(binary)=0.870  R=0.917  F1=0.893  auc=0.951
MISINFO class -> precision=0.870 recall=0.917 f1=0.893 (support=699)
confusion [rows=true, cols=pred] order=['non-misinformation', 'misinformation']:
    [590, 96]
    [58, 641]
wrong: 154/1385 (11.1%)

AfroXLMR-large-76L - TEST (untouched)
accuracy=0.875  P(binary)=0.850  R=0.915  F1=0.881  auc=0.942
MISINFO class -> precision=0.850 recall=0.915 f1=0.881 (support=704)
confusion [rows=true, cols=pred] order=['non-misinformation', 'misinformation']:
    [575, 114]
    [60, 644]
wrong: 174/1393 (12.5%)

------------------------------------------------------------------------
AfroXLMR-lg TEST · by language
------------------------------------------------------------------------
language                   n     acc      f1    %mis     auc
English                  468   0.910   0.912     52%   0.965
Sepedi                   466   0.854

In [ ]:
!pip uninstall -y torchao

In [ ]:
import traceback, transformers, accelerate, peft, torch
print(transformers.__version__, accelerate.__version__, peft.__version__, torch.__version__)
try:
    from transformers.trainer import Trainer   # bypasses the lazy wrapper
except Exception:
    traceback.print_exc()